# Face-Enabled Multipart Infill Evaluation

This notebook checks the face codec floor, face infill quality, and body-motion retention after appending face as the fifth 512x4 RVQ stream. The old and new transformers are evaluated on the same face-available validation clips.

In [ ]:
from pathlib import Path
import gc
import sys

import matplotlib.pyplot as plt
import pandas as pd
import torch
from IPython.display import display

cwd = Path.cwd().resolve()
if (cwd / 'motion_generation').is_dir():
    PROJECT_ROOT = cwd
elif cwd.name == 'notebooks' and cwd.parent.name == 'motion_generation':
    PROJECT_ROOT = cwd.parents[1]
else:
    raise RuntimeError(f'Run from the repository root or motion_generation/notebooks, not {cwd}')

MOTION_GENERATION_DIR = PROJECT_ROOT / 'motion_generation'
sys.path.insert(0, str(MOTION_GENERATION_DIR))
from scripts.train_audio_mask_multipart import discover_names, load_sequences, read_split_file
from utils.multipart_motion import MULTIMODAL_PART_ORDER, PART_ORDER
from utils.variable_c2f_evaluation import (
    InfillModelSpec,
    evaluate_face_codec_reconstruction,
    evaluate_model_windows,
    load_audio_motion_transformer,
    load_part_codecs,
    summarize_window_metrics,
)

DEVICE = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print('device:', DEVICE)

In [ ]:
DATA_DIR = PROJECT_ROOT / 'SuSuInterActs' / 'SuSuInterActs'
AUDIO_DIR = DATA_DIR / 'audio_features_hubert_layer9_fps10'
BODY_TOKEN_DIR = DATA_DIR / 'motion_token_data_multipart_512x4'
FACE_TOKEN_DIR = DATA_DIR / 'motion_face_token_data_multipart_512x4'
EVAL_SPLIT = DATA_DIR / 'split' / 'val_file_list.txt'
FACE_DIR = DATA_DIR / 'arkit_data'
CHECKPOINT_ROOT = PROJECT_ROOT / 'checkpoints'
RVQ_ROOT = CHECKPOINT_ROOT / 'multipart_rvqvae'

BODY_MODEL = InfillModelSpec(
    name='soft_recovery_body_only',
    checkpoint=CHECKPOINT_ROOT / 'mask_multipart_variable_c2f_soft_recovery_sf05_gap1_15',
    decoder='c2f',
    allowed_gaps=tuple(range(1, 16)),
)
FACE_MODEL = InfillModelSpec(
    name='soft_recovery_body_face',
    checkpoint=CHECKPOINT_ROOT / 'mask_multipart_face_variable_c2f_soft_recovery_sf05_stage2_gap1_15',
    decoder='c2f',
    allowed_gaps=tuple(range(1, 16)),
)

BODY_CHECKPOINTS = {
    part: RVQ_ROOT / f'rvq_{part}_512x4_bs256_cosine' / 'model' / 'best.pth'
    for part in PART_ORDER
}
FACE_CHECKPOINTS = {
    **BODY_CHECKPOINTS,
    'face': RVQ_ROOT / 'rvq_face_512x4_bs256_cosine' / 'model' / 'best.pth',
}
GAPS = [3, 7, 15]
WINDOWS_PER_CLIP = 1
BATCH_SIZE = 32
MAX_EVAL_CLIPS = None
OUT_DIR = MOTION_GENERATION_DIR / 'notebooks' / 'face_infill_eval_outputs'

## Shared Validation Census

In [ ]:
split_names = read_split_file(EVAL_SPLIT)
body_available = set(discover_names(BODY_TOKEN_DIR, AUDIO_DIR, split_names))
face_available = set(discover_names(FACE_TOKEN_DIR, AUDIO_DIR, split_names))
common_names = [name for name in split_names if name in body_available and name in face_available]
if MAX_EVAL_CLIPS is not None:
    common_names = common_names[:MAX_EVAL_CLIPS]
print('listed validation clips:', len(split_names))
print('body-token clips:', len(body_available))
print('face-token clips:', len(face_available))
print('paired clips:', len(common_names))

In [ ]:
body_codecs = load_part_codecs(BODY_CHECKPOINTS, DEVICE, part_order=PART_ORDER)
face_codecs = load_part_codecs(FACE_CHECKPOINTS, DEVICE, part_order=MULTIMODAL_PART_ORDER)

def load_paired(token_dir, part_order, codecs):
    codebook_size = next(iter(codecs.values())).codebook_size
    quantizers = next(iter(codecs.values())).num_quantizers
    sequences, stats = load_sequences(
        common_names, token_dir, AUDIO_DIR,
        codebook_size=codebook_size,
        num_tokens_per_frame=len(part_order) * quantizers,
        audio_fps=10.0,
        source_motion_fps_fallback=20.0,
        motion_token_fps_override=None,
        motion_token_unit_length_override=None,
    )
    return sequences, stats

body_sequences, body_stats = load_paired(BODY_TOKEN_DIR, PART_ORDER, body_codecs)
face_sequences, face_stats = load_paired(FACE_TOKEN_DIR, MULTIMODAL_PART_ORDER, face_codecs)
assert [x['name'] for x in body_sequences] == [x['name'] for x in face_sequences]
print('body:', body_stats)
print('body+face:', face_stats)

## Face Codec Floor

These errors compare the decoded ground-truth face tokens against raw ARKit coefficients. They are the lower bound for any infiller using this codec.

In [ ]:
codec_face = evaluate_face_codec_reconstruction(
    face_sequences, face_codecs, FACE_DIR, DEVICE, part_order=MULTIMODAL_PART_ORDER
)
codec_columns = [
    'face_mae', 'face_rmse', 'face_velocity_rmse',
    'face_lip_rmse', 'face_lip_velocity_rmse',
    'face_non_lip_rmse', 'face_non_lip_velocity_rmse',
]
display(codec_face[codec_columns].mean().to_frame('codec_floor').round(6))
OUT_DIR.mkdir(parents=True, exist_ok=True)
codec_face.to_csv(OUT_DIR / 'face_codec_reconstruction.csv', index=False)

## Paired Infill Evaluation

Body RMSE is directly comparable across both models. Face metrics exist only for the body+face model and are measured against the face codec target, so interpret them above the codec floor.

In [ ]:
runs = [
    (BODY_MODEL, body_sequences, body_codecs, PART_ORDER),
    (FACE_MODEL, face_sequences, face_codecs, MULTIMODAL_PART_ORDER),
]
frames = []
for spec, sequences, codecs, part_order in runs:
    print('loading', spec.name, spec.checkpoint)
    model = load_audio_motion_transformer(spec.checkpoint, DEVICE)
    frames.append(evaluate_model_windows(
        model, spec, sequences, codecs, GAPS,
        batch_size=BATCH_SIZE, device=DEVICE,
        windows_per_clip=WINDOWS_PER_CLIP, seed=42,
        slot_constrained=False, part_order=part_order,
    ))
    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
window_df = pd.concat(frames, ignore_index=True)
window_df.to_csv(OUT_DIR / 'paired_window_metrics.csv', index=False)
print('rows:', len(window_df))

In [ ]:
by_gap, macro = summarize_window_metrics(window_df)
body_columns = [
    'body_rmse', 'body_velocity_rmse',
    'part_upper_acc', 'part_lower_acc', 'part_feet_acc', 'part_hands_acc',
]
display(by_gap[[c for c in body_columns if c in by_gap.columns]].round(5))
face_columns = [
    'part_face_acc', 'face_rmse', 'face_velocity_rmse',
    'face_lip_rmse', 'face_lip_velocity_rmse',
    'face_non_lip_rmse', 'face_non_lip_velocity_rmse',
]
face_by_gap = (
    window_df[window_df['model'] == FACE_MODEL.name]
    .groupby('gap')[[c for c in face_columns if c in window_df.columns]]
    .mean()
)
display(face_by_gap.round(5))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
for model_name, group in by_gap.reset_index().groupby('model'):
    axes[0].plot(group['gap'], group['body_rmse'], marker='o', label=model_name)
    axes[1].plot(group['gap'], group['body_velocity_rmse'], marker='o', label=model_name)
axes[2].plot(face_by_gap.index, face_by_gap['face_lip_rmse'], marker='o', label='lip')
axes[2].plot(face_by_gap.index, face_by_gap['face_non_lip_rmse'], marker='o', label='non-lip')
axes[0].set_title('Body RMSE')
axes[1].set_title('Body velocity RMSE')
axes[2].set_title('Generated face RMSE')
for axis in axes:
    axis.set_xlabel('Gap token frames')
    axis.set_ylabel('Lower is better')
    axis.grid(alpha=0.25)
    axis.legend()
fig.tight_layout()
plt.show()

## Decision Rule

Accept the combined model only if face errors remain reasonably close to the codec floor and body RMSE does not regress materially on the paired subset. Motion FID/R@K should still be rerun with the existing full-clip evaluator before reporting the final model.